# Part 2: Mathematical Foundations — MDPs & The Bellman Equation

**Course**: Reinforcement Learning for AI/ML Engineers  
**Duration**: 1.5 Hours  
**Instructor**: Mehdi Lotfinejad  

---

## Learning Objectives

By the end of this section, you will be able to:

1. **Understand** what a Markov Decision Process (MDP) is through practical examples
2. **Build** a complete MDP in Python and explore its components
3. **Compute** value functions — how good is a state or action?
4. **Apply** the Bellman Equation to find optimal strategies automatically
5. **Implement** Value Iteration and Policy Iteration from scratch
6. **Experiment** with the discount factor and see how it changes agent behavior

---

## 1. Why Do We Need Math?

In Part 1, our Q-Learning agent worked great on the Grid World. But we had some unanswered questions:

- **How do we know** the agent found the *best* path, not just a *good* one?
- **Can we prove** the algorithm will always find the optimal solution?
- **Can we calculate** the exact best strategy without trial-and-error?

The answer to all three is **YES** — using **Markov Decision Processes (MDPs)** and the **Bellman Equation**.

> Think of it this way: Q-Learning is like learning to cook by experimenting. MDPs and Bellman are like having the recipe book — you can calculate the perfect dish without guessing.

### What We'll Build Today

```
Section 2: The Markov Property     →  "The present is all you need"
Section 3: MDPs                    →  Build a complete decision problem in Python
Section 4: Policies                →  Compare different strategies head-to-head
Section 5: Value Functions         →  Score every state: "how good is it to be here?"
Section 6: The Bellman Equation    →  THE key equation that solves everything
Section 7: Value Iteration         →  Algorithm that finds the BEST strategy automatically
Section 8: Discount Factor         →  How "patient" should the agent be?
Section 9: Grid World MDP          →  Put it all together on a bigger problem
```

## 2. The Markov Property — "The Present is All You Need"

Before we build MDPs, we need one key concept: the **Markov Property**.

### The Simple Version

> **The Markov Property** says: to predict the future, you only need to know the **current state** — not the entire history of how you got there.

### Everyday Examples

| Scenario | Markov? | Why? |
|---|---|---|
| **Chess board** | ✅ Yes | The current position tells you everything. It doesn't matter how you got there. |
| **Your GPS location + speed** | ✅ Yes | Your current position and speed are enough to predict where you'll be next. |
| **"What's the stock price today?"** | ❌ No | Just today's price isn't enough — you need the trend, volume, history. |
| **Stock price + trend + volume** | ✅ Yes | If you include enough info in the "state," it becomes Markov! |

### Why Does This Matter?

If a problem has the Markov Property, we can use all the powerful MDP tools. The trick is **designing the right state** — include enough information so the present captures everything relevant.

> **Pro Tip**: If your RL agent isn't learning well, try enriching the state with more information!

## 3. Markov Decision Process — The Complete RL Framework

An **MDP** is simply a formal way to describe any RL problem. Every MDP has exactly 5 components:

| Component | Symbol | Plain English |
|---|---|---|
| **States** | S | All the situations the agent can be in |
| **Actions** | A | All the moves the agent can make |
| **Transitions** | P(s'\|s,a) | "If I'm in state s and do action a, where do I end up?" |
| **Rewards** | R(s,a,s') | "How much reward do I get?" |
| **Discount** | γ (gamma) | "How much do I care about future vs. immediate rewards?" |

### 3.1 Let's Build One: The Student MDP

Here's a fun example — a student deciding whether to study or procrastinate:

```
┌─────────────────────────────────────────────────────────────┐
│                    THE STUDENT MDP                           │
│                                                             │
│              Study(-2)        Study(-2)        Study(+10)   │
│  ┌─────────┐ ──────► ┌─────────┐ ──────► ┌─────────┐ ──► PASS!│
│  │ CLASS 1 │         │ CLASS 2 │         │ CLASS 3 │       │
│  └────┬────┘         └────┬────┘         └────┬────┘       │
│       │ Facebook(-1)      │ Sleep(0)          │ Pub(+1)    │
│       ▼                   ▼                   ▼            │
│  ┌──────────┐       ┌─────────┐    40%→Class1              │
│  │ FACEBOOK │       │  SLEEP  │    40%→Class2              │
│  │  (loop)  │       │  (end)  │    20%→Class3              │
│  └──────────┘       └─────────┘                            │
│                                                             │
│  Key insight: Studying is painful now (-2) but pays off     │
│  later (+10). Going to the Pub is fun now (+1) but risky!   │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np

# ============================================================
# The Student MDP - Built in Python
# ============================================================
class StudentMDP:
    def __init__(self):
        self.states = ['Class1', 'Class2', 'Class3', 'Facebook', 'Sleep', 'Pass']
        self.terminal_states = {'Sleep', 'Pass'}
        
        # What actions can you take in each state?
        self.actions = {
            'Class1':   ['Study', 'Facebook'],
            'Class2':   ['Study', 'Sleep'],
            'Class3':   ['Study', 'Pub'],
            'Facebook': ['Facebook', 'Quit'],
            'Sleep':    [],    # Game over
            'Pass':     [],    # Game over (you won!)
        }
        
        # What happens when you take each action?
        # Format: (probability, next_state, reward)
        self.transitions = {
            ('Class1', 'Study'):      [(1.0, 'Class2', -2)],     # Studying is hard...
            ('Class1', 'Facebook'):   [(1.0, 'Facebook', -1)],   # Distraction!
            ('Class2', 'Study'):      [(1.0, 'Class3', -2)],     # More studying...
            ('Class2', 'Sleep'):      [(1.0, 'Sleep', 0)],       # Give up
            ('Class3', 'Study'):      [(1.0, 'Pass', +10)],      # BIG payoff!
            ('Class3', 'Pub'):        [(0.4, 'Class1', +1),      # Fun but risky:
                                       (0.4, 'Class2', +1),      #   might end up
                                       (0.2, 'Class3', +1)],     #   back in class!
            ('Facebook', 'Facebook'): [(1.0, 'Facebook', -1)],   # Doom scrolling...
            ('Facebook', 'Quit'):     [(1.0, 'Class1', 0)],      # Back to work!
        }
    
    def get_transitions(self, state, action):
        return self.transitions.get((state, action), [])
    
    def get_actions(self, state):
        return self.actions.get(state, [])
    
    def is_terminal(self, state):
        return state in self.terminal_states

# Explore the MDP
mdp = StudentMDP()

print("=" * 55)
print("THE STUDENT MDP")
print("=" * 55)

print("\nStates and available actions:")
for state in mdp.states:
    actions = mdp.get_actions(state)
    status = "[GAME OVER]" if not actions else str(actions)
    print(f"  {state:12s}  Actions: {status}")

print("\nWhat happens with each action:")
print("-" * 55)
for (state, action), transitions in mdp.transitions.items():
    for prob, next_state, reward in transitions:
        arrow = f"{state} --[{action}]--> {next_state}"
        print(f"  {arrow:<40s} P={prob:.0%}  R={reward:+d}")

### 3.2 Key Concepts: Deterministic vs. Stochastic

Notice the **Pub** action in Class 3:
- 40% chance you end up back in Class 1
- 40% chance you end up in Class 2  
- 20% chance you stay in Class 3

This is a **stochastic** (random) transition. Not all actions have guaranteed outcomes! This is what makes RL challenging and realistic — just like real life, outcomes are uncertain.

## 4. Policies — Comparing Different Strategies

A **policy** is simply the agent's strategy: *"When I'm in this state, I do this action."*

Let's define three different student strategies and see which one works best!

In [ ]:
# ============================================================
# Three Different Student Strategies (Policies)
# ============================================================

# Strategy A: "The Grinder" - always studies
policy_grinder = {
    'Class1': {'Study': 1.0}, 'Class2': {'Study': 1.0},
    'Class3': {'Study': 1.0}, 'Facebook': {'Quit': 1.0},
}

# Strategy B: "The Slacker" - avoids work at all costs
policy_slacker = {
    'Class1': {'Facebook': 1.0}, 'Class2': {'Sleep': 1.0},
    'Class3': {'Pub': 1.0}, 'Facebook': {'Facebook': 1.0},
}

# Strategy C: "The Balanced" - studies most of the time
policy_balanced = {
    'Class1': {'Study': 0.7, 'Facebook': 0.3},
    'Class2': {'Study': 0.8, 'Sleep': 0.2},
    'Class3': {'Study': 0.6, 'Pub': 0.4},
    'Facebook': {'Quit': 0.5, 'Facebook': 0.5},
}

def simulate_policy(mdp, policy, n_episodes=10000, gamma=0.9):
    outcomes = {'Pass': 0, 'Sleep': 0, 'Stuck': 0}
    total_returns = []
    for _ in range(n_episodes):
        state = 'Class1'
        total_return = 0
        discount = 1.0
        for step in range(200):
            if mdp.is_terminal(state):
                outcomes['Pass' if state == 'Pass' else 'Sleep'] += 1
                break
            action_probs = policy.get(state, {})
            if not action_probs:
                outcomes['Stuck'] += 1
                break
            action = np.random.choice(list(action_probs.keys()), 
                                       p=list(action_probs.values()))
            transitions = mdp.get_transitions(state, action)
            probs = [t[0] for t in transitions]
            idx = np.random.choice(len(transitions), p=probs)
            _, next_state, reward = transitions[idx]
            total_return += discount * reward
            discount *= gamma
            state = next_state
        total_returns.append(total_return)
    return np.mean(total_returns), outcomes

print("=" * 60)
print("STRATEGY SHOWDOWN: Which Student Does Best?")
print("=" * 60)

strategies = {
    "The Grinder (always study)":  policy_grinder,
    "The Slacker (avoid work)":    policy_slacker,
    "The Balanced (mostly study)": policy_balanced,
}

for name, policy in strategies.items():
    avg_return, outcomes = simulate_policy(mdp, policy)
    total = sum(outcomes.values())
    print(f"\n  {name}")
    print(f"    Average Score:  {avg_return:+.2f}")
    print(f"    Pass Rate:      {outcomes['Pass']/total*100:.1f}%")
    print(f"    Give-up Rate:   {outcomes['Sleep']/total*100:.1f}%")

print("\n" + "=" * 60)
print("The Grinder wins! But is it the BEST possible strategy?")
print("Let's find out using the Bellman Equation...")

## 5. Value Functions — "How Good Is This State?"

A **value function** answers the question: *"If I'm in this state and follow my strategy, how much total reward can I expect?"*

### Two Types of Value Functions

**V(s) — State Value**: "How good is it to BE in state s?"
> Example: V(Class3) = 8.1 means "if you're in Class 3 and follow the strategy, you can expect about 8.1 total reward"

**Q(s, a) — Action Value**: "How good is it to DO action a in state s?"
> Example: Q(Class3, Study) = 10.0 but Q(Class3, Pub) = 3.5 means "studying in Class 3 is way better than going to the pub"

### Let's Calculate V(s) by Simulation

The simplest way: play thousands of games, track the total reward from each state, and average them.

In [ ]:
# ============================================================
# Calculate V(s) by Playing Lots of Games (Monte Carlo)
# ============================================================
def estimate_values(mdp, policy, gamma=0.9, n_episodes=50000):
    state_returns = {s: [] for s in mdp.states}
    
    for _ in range(n_episodes):
        # Start from a random state
        non_terminal = [s for s in mdp.states if not mdp.is_terminal(s)]
        start = np.random.choice(non_terminal)
        
        # Play one episode, recording what happens
        episode = []
        state = start
        for step in range(200):
            if mdp.is_terminal(state):
                break
            action_probs = policy.get(state, {})
            if not action_probs:
                break
            action = np.random.choice(list(action_probs.keys()),
                                       p=list(action_probs.values()))
            transitions = mdp.get_transitions(state, action)
            probs = [t[0] for t in transitions]
            idx = np.random.choice(len(transitions), p=probs)
            _, next_state, reward = transitions[idx]
            episode.append((state, reward))
            state = next_state
        
        # Calculate total discounted reward for each state visited
        G = 0
        visited = set()
        for state_ep, reward in reversed(episode):
            G = reward + gamma * G
            if state_ep not in visited:
                state_returns[state_ep].append(G)
                visited.add(state_ep)
    
    V = {s: (np.mean(state_returns[s]) if state_returns[s] else 0.0) for s in mdp.states}
    return V

print("=" * 55)
print("STATE VALUES V(s) FOR EACH STRATEGY")
print("=" * 55)

for name, policy in strategies.items():
    V = estimate_values(mdp, policy)
    print(f"\n  {name}")
    for state in mdp.states:
        bar_len = max(0, int((V[state] + 5) * 1.5))
        bar = chr(9608) * bar_len
        print(f"    {state:<12} V = {V[state]:>+6.2f}  {bar}")

print("\nHigher V(s) = better expected outcome from that state")
print("The Grinder has the highest values in every state!")

## 6. The Bellman Equation — The Most Important Equation in RL

The Monte Carlo approach (simulating thousands of games) works, but it's slow and only gives approximate answers. The **Bellman Equation** gives us the **exact** answer!

### The Idea (No Scary Math!)

The value of a state can be broken down into two parts:

```
Value of THIS state = Immediate Reward + Discounted Value of NEXT state
```

It's like asking "How much money will I make on this job?" You can answer:

```
Total career earnings = This year's salary + (discount * remaining career earnings)
```

### The Actual Equation

For a given strategy (policy), the value of state s is:

$$V(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma \cdot V(s') \right]$$

**Don't panic!** In plain English this says:

> V(s) = For each action I might take (weighted by my strategy), look at each place I might end up, and add up: the immediate reward + discounted future value.

### Let's Implement It!

Instead of simulating thousands of games, we'll solve it directly with a loop that keeps updating until the values stabilize.

In [ ]:
# ============================================================
# Policy Evaluation: Calculate EXACT V(s) Using Bellman Equation
# ============================================================
def policy_evaluation(mdp, policy, gamma=0.9, threshold=1e-6):
    # Start with V(s) = 0 for all states
    V = {s: 0.0 for s in mdp.states}
    
    iteration = 0
    while True:
        iteration += 1
        max_change = 0   # Track biggest change this round
        
        for state in mdp.states:
            if mdp.is_terminal(state):
                continue   # Terminal states always have value 0
            
            action_probs = policy.get(state, {})
            if not action_probs:
                continue
            
            # Apply Bellman equation
            new_value = 0
            for action, action_prob in action_probs.items():
                for prob, next_state, reward in mdp.get_transitions(state, action):
                    new_value += action_prob * prob * (reward + gamma * V[next_state])
            
            max_change = max(max_change, abs(new_value - V[state]))
            V[state] = new_value
        
        # Stop when values barely change
        if max_change < threshold:
            print(f"  Converged in {iteration} iterations (changes < {threshold})")
            break
    
    return V

print("=" * 60)
print("EXACT VALUES via Bellman Equation")
print("=" * 60)

all_V = {}
for name, policy in strategies.items():
    print(f"\n  {name}:")
    V = policy_evaluation(mdp, policy, gamma=0.9)
    all_V[name] = V
    for state in mdp.states:
        print(f"    {state:<12} V = {V[state]:>+8.4f}")

In [ ]:
# ============================================================
# Compare: Exact Bellman vs. Approximate Monte Carlo
# ============================================================
V_exact = all_V["The Grinder (always study)"]
V_approx = estimate_values(mdp, policy_grinder, gamma=0.9, n_episodes=100000)

print("=" * 65)
print("EXACT (Bellman) vs. APPROXIMATE (Monte Carlo) - Grinder Policy")
print("=" * 65)
print(f"\n  {'State':<12} {'Bellman (Exact)':>16} {'MC (100k games)':>16} {'Error':>8}")
print(f"  {'-'*54}")
for state in mdp.states:
    diff = abs(V_exact[state] - V_approx[state])
    print(f"  {state:<12} {V_exact[state]:>+16.4f} {V_approx[state]:>+16.4f} {diff:>8.4f}")

print(f"\n  Bellman gives the EXACT answer instantly!")
print(f"  Monte Carlo needed 100,000 games and still has small errors.")

## 7. Value Iteration — Finding the BEST Strategy Automatically

So far we've only *evaluated* strategies we chose ourselves. But what if we want the computer to **find the best strategy on its own**?

That's what **Value Iteration** does! Instead of averaging over the actions in our policy, it takes the **BEST action** at each state:

```
Regular Bellman:   V(s) = average over all actions (weighted by policy)
Optimal Bellman:   V*(s) = BEST action's value
```

The only change is replacing "average" with "max":

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R + \gamma \cdot V^*(s') \right]$$

In [ ]:
# ============================================================
# Value Iteration: Find the OPTIMAL Strategy Automatically!
# ============================================================
def value_iteration(mdp, gamma=0.9, threshold=1e-6):
    V = {s: 0.0 for s in mdp.states}
    
    iteration = 0
    while True:
        iteration += 1
        max_change = 0
        
        for state in mdp.states:
            if mdp.is_terminal(state):
                continue
            actions = mdp.get_actions(state)
            if not actions:
                continue
            
            # Find the BEST action (this is the key difference!)
            action_values = []
            for action in actions:
                q = sum(prob * (reward + gamma * V[next_s]) 
                        for prob, next_s, reward in mdp.get_transitions(state, action))
                action_values.append(q)
            
            best_value = max(action_values)
            max_change = max(max_change, abs(best_value - V[state]))
            V[state] = best_value
        
        if max_change < threshold:
            print(f"  Converged in {iteration} iterations")
            break
    
    # Extract the optimal policy: pick the best action in each state
    optimal_policy = {}
    for state in mdp.states:
        if mdp.is_terminal(state):
            continue
        actions = mdp.get_actions(state)
        if not actions:
            continue
        best_action, best_value = None, float('-inf')
        for action in actions:
            q = sum(prob * (reward + gamma * V[next_s])
                    for prob, next_s, reward in mdp.get_transitions(state, action))
            if q > best_value:
                best_value = q
                best_action = action
        optimal_policy[state] = best_action
    
    return V, optimal_policy

print("=" * 55)
print("VALUE ITERATION: Finding the Optimal Strategy")
print("=" * 55)

V_star, best_policy = value_iteration(mdp, gamma=0.9)

print(f"\nOptimal Values V*(s):")
for state in mdp.states:
    print(f"  {state:<12} V* = {V_star[state]:>+8.4f}")

print(f"\nOptimal Strategy:")
for state, action in best_policy.items():
    print(f"  In {state:<12} -> always {action}")

print(f"\nThe computer discovered: ALWAYS STUDY is optimal!")
print(f"(and quit Facebook immediately if you end up there)")

In [ ]:
# ============================================================
# Proof: Compare All Strategies Against the Optimal
# ============================================================
print("=" * 65)
print("FINAL COMPARISON: All Strategies vs. Optimal")
print("=" * 65)

print(f"\n{'State':<12} {'OPTIMAL':>10} {'Grinder':>10} {'Slacker':>10} {'Balanced':>10}")
print("-" * 55)

for state in mdp.states:
    v_opt = V_star[state]
    v_gri = all_V["The Grinder (always study)"][state]
    v_sla = all_V["The Slacker (avoid work)"][state]
    v_bal = all_V["The Balanced (mostly study)"][state]
    match = " <-best" if abs(v_opt - v_gri) < 0.01 else ""
    print(f"{state:<12} {v_opt:>+10.4f} {v_gri:>+10.4f}{match} {v_sla:>+10.4f} {v_bal:>+10.4f}")

print(f"\nThe Grinder matches the optimal in every state!")
print(f"Value Iteration mathematically PROVED it's the best strategy.")

## 8. The Discount Factor γ — How Patient Is Your Agent?

The discount factor $\gamma$ (gamma) controls how much the agent cares about **future rewards** vs. **immediate rewards**.

| γ Value | Agent Personality | Analogy |
|---|---|---|
| **γ = 0** | "I only care about RIGHT NOW" | A toddler wanting candy |
| **γ = 0.5** | "I'll plan a little ahead" | Planning for next week |
| **γ = 0.9** | "I think long-term" | Saving for retirement |
| **γ = 0.99** | "I'm planning decades ahead" | Building generational wealth |

### Let's See How γ Changes the Optimal Strategy

In [ ]:
# ============================================================
# How Discount Factor Changes the Optimal Strategy
# ============================================================
print("=" * 65)
print("HOW DOES PATIENCE (gamma) AFFECT THE BEST STRATEGY?")
print("=" * 65)

for gamma in [0.0, 0.1, 0.5, 0.9, 0.99]:
    V, policy = value_iteration(mdp, gamma=gamma)
    actions = ", ".join([f"{s}->{a}" for s, a in policy.items()])
    print(f"\n  gamma = {gamma:.2f} ({'myopic' if gamma < 0.3 else 'patient' if gamma > 0.7 else 'moderate'}):")
    print(f"    Strategy: {actions}")
    print(f"    V*(Class1) = {V['Class1']:+.4f}")

print("\n" + "=" * 65)
print("  gamma near 0: Agent is impatient, may avoid short-term pain (studying)")
print("  gamma near 1: Agent is patient, willing to study now for big payoff later")
print("\n  Takeaway: gamma is one of the most important hyperparameters in RL!")

## 9. Hands-On: Complete Grid World MDP

Let's apply everything to a bigger, more visual problem — a 4x4 grid with **stochastic transitions** (actions don't always go where you want!).

```
┌────┬────┬────┬────┐
│ S  │    │    │    │    S = Start    G = Goal (+1)
├────┼────┼────┼────┤    X = Wall (can't enter)
│    │ XX │    │    │    
├────┼────┼────┼────┤    STOCHASTIC: When you try to move:
│    │    │    │    │      80% → go where you want
├────┼────┼────┼────┤      10% → slip left
│    │    │    │ G  │      10% → slip right
└────┴────┴────┴────┘    Each step costs -0.04
```

In [ ]:
# ============================================================
# 4x4 Stochastic Grid World MDP
# ============================================================
class GridWorldMDP:
    def __init__(self, step_reward=-0.04):
        self.rows, self.cols = 4, 4
        self.walls = {(1, 1)}
        self.goal = (3, 3)
        self.goal_reward = 1.0
        self.step_reward = step_reward
        self.states = [(r, c) for r in range(4) for c in range(4) if (r,c) not in self.walls]
        self.terminal_states = {self.goal}
        self.actions = ['Up', 'Down', 'Left', 'Right']
        self.effects = {'Up':(-1,0), 'Down':(1,0), 'Left':(0,-1), 'Right':(0,1)}
        self.perp = {'Up':['Left','Right'], 'Down':['Left','Right'],
                     'Left':['Up','Down'], 'Right':['Up','Down']}
    
    def _move(self, state, action):
        dr, dc = self.effects[action]
        nr, nc = max(0, min(3, state[0]+dr)), max(0, min(3, state[1]+dc))
        return state if (nr,nc) in self.walls else (nr,nc)
    
    def get_transitions(self, state, action):
        if state in self.terminal_states:
            return []
        trans = {}
        intended = self._move(state, action)
        trans[intended] = trans.get(intended, 0) + 0.8
        for pa in self.perp[action]:
            ps = self._move(state, pa)
            trans[ps] = trans.get(ps, 0) + 0.1
        return [(p, ns, self.goal_reward if ns == self.goal else self.step_reward) 
                for ns, p in trans.items()]
    
    def get_actions(self, state):
        return [] if state in self.terminal_states else self.actions
    
    def is_terminal(self, state):
        return state in self.terminal_states

def solve_grid(gw, gamma=0.9):
    V = {s: 0.0 for s in gw.states}
    for _ in range(1000):
        delta = 0
        for s in gw.states:
            if gw.is_terminal(s): continue
            vals = [sum(p*(r+gamma*V.get(ns,0)) for p,ns,r in gw.get_transitions(s,a)) 
                    for a in gw.actions]
            new_v = max(vals)
            delta = max(delta, abs(new_v - V[s]))
            V[s] = new_v
        if delta < 1e-6: break
    policy = {}
    for s in gw.states:
        if gw.is_terminal(s):
            policy[s] = 'GOAL'
            continue
        best_a, best_v = None, float('-inf')
        for a in gw.actions:
            q = sum(p*(r+gamma*V.get(ns,0)) for p,ns,r in gw.get_transitions(s,a))
            if q > best_v: best_v, best_a = q, a
        policy[s] = best_a
    return V, policy

# Solve it!
gw = GridWorldMDP()
V_gw, policy_gw = solve_grid(gw)

arrows = {'Up': chr(8593), 'Down': chr(8595), 'Left': chr(8592), 'Right': chr(8594), 'GOAL': chr(9733)}

print("=" * 50)
print("OPTIMAL VALUES AND POLICY")
print("=" * 50)
print("\nValues V*(s) and optimal actions:")
for r in range(4):
    val_row = "  "
    act_row = "  "
    for c in range(4):
        if (r,c) in gw.walls:
            val_row += " [WALL]  "
            act_row += "  WALL   "
        elif (r,c) == gw.goal:
            val_row += " [GOAL]  "
            act_row += f"   {arrows['GOAL']}    "
        else:
            val_row += f" {V_gw[(r,c)]:+.3f}  "
            act_row += f"   {arrows[policy_gw[(r,c)]]}    "
    print(val_row)
    print(act_row)
    print()

print("The agent learned to navigate around the wall to reach the goal!")

### ✏️ Exercise 2: Experiment with Parameters

Change the step cost and discount factor to see how the optimal policy changes!

In [ ]:
# ============================================================
# Exercise 2: How Do Parameters Change the Optimal Path?
# ============================================================
arrows = {'Up': chr(8593), 'Down': chr(8595), 'Left': chr(8592), 'Right': chr(8594), 'GOAL': chr(9733)}

print("EXPERIMENT: Different step costs")
print("=" * 55)

for cost in [-0.01, -0.04, -0.1, -0.5]:
    gw_exp = GridWorldMDP(step_reward=cost)
    V, policy = solve_grid(gw_exp, gamma=0.9)
    print(f"\n  Step cost = {cost:+.2f}  (V* at start = {V[(0,0)]:+.4f})")
    for r in range(4):
        row = "    "
        for c in range(4):
            if (r,c) in gw_exp.walls: row += chr(9619) + " "
            else: row += arrows[policy[(r,c)]] + " "
        print(row)

print("\n" + "=" * 55)
print("  Small cost (-0.01): Agent may wander -- no urgency")
print("  Large cost (-0.50): Agent rushes to goal ASAP")

print("\n\nEXPERIMENT: Different patience levels (gamma)")
print("=" * 55)

gw_exp2 = GridWorldMDP()
for gamma in [0.1, 0.5, 0.9, 0.99]:
    V, policy = solve_grid(gw_exp2, gamma=gamma)
    print(f"\n  gamma = {gamma}  (V* at start = {V[(0,0)]:+.4f})")
    for r in range(4):
        row = "    "
        for c in range(4):
            if (r,c) in gw_exp2.walls: row += chr(9619) + " "
            else: row += arrows[policy[(r,c)]] + " "
        print(row)

## 10. Summary & Key Takeaways

### What We Learned Today

✅ **Markov Property**: The current state is all you need to make decisions  

✅ **MDP**: The formal framework — States, Actions, Transitions, Rewards, Discount  

✅ **Policies**: Different strategies can be compared by their value functions  

✅ **Value Functions**: V(s) = "how good is this state?", Q(s,a) = "how good is this action?"  

✅ **Bellman Equation**: Breaks down value into immediate reward + discounted future value  

✅ **Value Iteration**: Finds the optimal strategy automatically!  

✅ **Discount Factor**: Controls how much the agent cares about future vs. now  

---

### Plain English Cheat Sheet

| Concept | Plain English |
|---|---|
| **MDP** | A formal description of a decision problem |
| **Transition P(s'\|s,a)** | "If I do action a in state s, where might I end up?" |
| **Value V(s)** | "How much total reward can I expect from state s?" |
| **Q-Value Q(s,a)** | "How much total reward if I do action a in state s?" |
| **Bellman Equation** | V(here) = immediate reward + discounted V(next) |
| **Value Iteration** | Keep updating values until they stabilize → optimal strategy |
| **Discount γ** | How patient the agent is (0 = impatient, 0.99 = very patient) |

---

### Coming Up in Part 3: Key RL Algorithms

We'll implement the big algorithms:
1. **Dynamic Programming** — solve MDPs when you know the rules
2. **Monte Carlo** — learn from complete episodes (no model needed!)
3. **Q-Learning** — the famous algorithm (learn without knowing the rules!)
4. **SARSA** — a safer alternative to Q-Learning
5. See how they compare on real environments!

---

*© 2025 Reinforcement Learning for AI/ML Engineers — Mehdi Lotfinejad*

## 📋 Quick Reference Card

```
┌──────────────────────────────────────────────────────────────────┐
│           MDP & BELLMAN EQUATION CHEAT SHEET                     │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  MDP = (States, Actions, Transitions, Rewards, Discount)         │
│                                                                  │
│  MARKOV PROPERTY:                                                │
│  "The present state is all you need to predict the future"       │
│                                                                  │
│  VALUE FUNCTIONS:                                                │
│  • V(s)   = "How good is this state?"                            │
│  • Q(s,a) = "How good is this action in this state?"             │
│                                                                  │
│  BELLMAN EQUATION:                                               │
│  V(s) = immediate reward + discount * V(next state)              │
│                                                                  │
│  ALGORITHMS:                                                     │
│  • Policy Evaluation  = Calculate V(s) for a given strategy      │
│  • Value Iteration    = Find the BEST strategy automatically     │
│                                                                  │
│  DISCOUNT FACTOR gamma:                                          │
│  • 0.0  = only care about now                                    │
│  • 0.9  = think long-term                                        │
│  • 0.99 = very far-sighted                                       │
│                                                                  │
│  KEY INSIGHT:                                                    │
│  Bellman gives EXACT answers; Monte Carlo only approximates.     │
│  But Bellman needs to know the rules; MC learns by playing.      │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```